# Khám phá dữ liệu

**Vai trò:** ghi lại EDA và các quyết định tiền xử lý cho bộ dữ liệu bệnh tim.

## Các bước thực hiện
1. Ghi nguồn, phiên bản, mô tả dữ liệu và cách đọc file vào.
2. Kiểm tra kích thước, kiểu dữ liệu, missing values, dòng trùng và miền giá trị.
3. Khảo sát phân phối biến số/biến phân loại và quan hệ giữa các thuộc tính.
4. Xác định biến đầu vào cho clustering và giữ nhãn `target` riêng.
5. Ghi rõ quy trình làm sạch, harmonize encoding, imputation và metric khoảng cách.

Notebook này đang minh họa quy trình thực tế của dự án: đọc dữ liệu từ `src/data.py`, chuẩn hóa encoding UCI, xử lý giá trị bất thường và NaN, rồi chuẩn bị input cho thuật toán phân cụm.

## 1.Nguồn dữ liệu

- **Tệp:** `data/raw/heart_disease_whole.csv` (dữ liệu gốc, không chỉnh sửa; thuộc `data/raw/`, thường được giữ ngoài Git).
- **Nguồn / phiên bản / ngày tải / giấy phép:** Bộ dữ liệu bệnh tim UCI/Heart Disease, tập đa nguồn (multi-source) được chọn theo `source = 'uci_multi_cleveland'` để giữ encoding thống nhất.
- **Mỗi dòng:** hồ sơ của một bệnh nhân gồm 13 đặc trưng + `target` (0: không bệnh tim, 1: có bệnh tim).
- Exact upstream URL, release/version, retrieval date and license are not recorded in this repository; verify them from the actual source before submitting the report.


### Environment setup

This notebook uses pandas, numpy, scikit-learn, matplotlib, seaborn and kmedoids. Install once from a terminal if needed:

```bash
python -m pip install pandas numpy scikit-learn matplotlib seaborn kmedoids
```


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import importlib
import src.data
importlib.reload(src.data)   # bắt buộc nếu không restart kernel

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import (
    load_heart_data, drop_duplicate_rows, replace_abnormal_with_nan,
    impute_mode, impute_median, split_features_target, data_quality_report,
    NUMERIC_COLS, CATEGORICAL_COLS, BINARY_COLS, TARGET_COL,
)
from src.preprocessing import preprocess_features, gower_distance_matrix, save_processed
from src.evaluation import run_and_save_metrics

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

## 2. Kích thước, kiểu dữ liệu, thiếu, trùng, giá trị bất thường

In [ ]:
df_raw = load_heart_data(
    ROOT / "data" / "raw" / "heart_disease_whole.csv",
    sources=["uci_multi_cleveland"],
)
print(df_raw.shape)
df_raw.dtypes

In [ ]:
report = data_quality_report(df_raw)
report

**Ghi chú quyết định (dữ liệu raw UCI / multi-source)**
- `load_heart_data()` đọc file `heart_disease_whole.csv`, rồi filter theo `source` khi cần; khi `source` là UCI multi Cleveland, dữ liệu đã ở dạng UCI tương thích.
- Một số source Kaggle/0-based được harmonize về chuẩn UCI 1-based (`cp`, `slope`, `thal`) trước khi xử lý.
- Các mã bất thường như `thal=0/1`, `slope=0`, `ca=4` được đổi thành NaN, sau đó impute bằng `impute_mode()` hoặc `impute_median()`.
- Dedup chạy trên 13 đặc trưng + `target` sau khi bỏ cột `source`; `target` được giữ riêng và không đưa vào ma trận clustering.
- Trong phần tiền xử lý cho mô hình, `preprocess_features()` dùng `StandardScaler` cho numeric, `OneHotEncoder` cho categorical và giữ nguyên các biến nhị phân.

In [ ]:
# 1. Đánh dấu TẤT CẢ mã sai thành NaN (không chỉ ca==4)
df_nan = replace_abnormal_with_nan(df_raw)

# 2. Dedup (source đã bị bỏ ở load)
df_dedup = drop_duplicate_rows(df_nan)

# 3. Điền mode cho categorical, median cho numeric
df, fill_cat = impute_mode(df_dedup)
df, fill_num = impute_median(df)

print(f"Raw: {len(df_raw)} | Sau dedup: {len(df)} | Bỏ: {len(df_raw) - len(df)}")
print(f"Fill categorical: {fill_cat}")
print(f"Fill numeric: {fill_num}")
print(f"NaN còn lại: {df.isna().sum().sum()}")
assert not df[NUMERIC_COLS + CATEGORICAL_COLS + BINARY_COLS].isna().any().any(), "Missing feature values remain"
assert df[TARGET_COL].notna().all(), "Missing target values remain"

df.describe().T

## 3. Phân phối biến

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.2))
for ax, col in zip(axes, NUMERIC_COLS):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = CATEGORICAL_COLS + BINARY_COLS
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
df[TARGET_COL].value_counts().rename("n").to_frame().assign(tỉ_lệ=lambda t: (t["n"] / t["n"].sum()).round(3))

## 4. Quan hệ giữa các biến số

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[NUMERIC_COLS].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Tương quan (biến số)")
plt.show()

## 5. Quyết định tiền xử lý

| Nhóm | Cột | Xử lý |
|---|---|---|
| Biến số | age, trestbps, chol, thalach, oldpeak | `StandardScaler` |
| Biến phân loại | cp, restecg, slope, ca, thal | `OneHotEncoder` |
| Biến nhị phân | sex, fbs, exang | giữ nguyên |
| Nhãn | target | tách riêng, **không** đưa vào phân cụm; chỉ kiểm tra sau phân cụm |

- **Metric chính:** Manhattan trên ma trận đã tiền xử lý.
- **Đối chứng:** Gower tính trên 13 cột gốc (số: hiệu / khoảng giá trị; phân loại & nhị phân: khớp/không khớp) → `KMedoids(metric='precomputed')`.
- **Mô hình:** `kmedoids.KMedoids via src/library_model.py (method="pam", init="build", random_state=42)`, chọn k trong 2–8 bằng silhouette.

The current metrics pipeline runs the library adapter. Do not describe this cell as comparing against the from-scratch implementation unless that implementation is explicitly run.

In [ ]:
X, y = split_features_target(df)
X_mat, feature_names, pre = preprocess_features(X)
print("Ma trận đặc trưng:", X_mat.shape)
print(feature_names)

## 6. Lưu dữ liệu đã tiền xử lý vào `data/processed/`


In [ ]:
D_gower = gower_distance_matrix(X)
saved = save_processed(df, X_mat, feature_names, D_gower, ROOT / "data" / "processed")
for name, path in saved.items():
    print(f"{name}: {path}")

In [ ]:
# Kiểm tra file đã lưu đọc lại được và khớp số dòng.
import numpy as np
chk_df = pd.read_csv(saved["dedup"]); chk_X = pd.read_csv(saved["X"])
chk_y = pd.read_csv(saved["target"]); chk_D = np.load(saved["gower"])
print("heart_dedup:", chk_df.shape, "| X_manhattan:", chk_X.shape, "| target:", chk_y.shape, "| gower:", chk_D.shape)
assert len(chk_df) == len(chk_X) == len(chk_y) == chk_D.shape[0] == chk_D.shape[1]
print("Khớp số dòng:", len(chk_df))

## 7. Tính và lưu metric vào `outputs/metrics/`
Chạy k = 2–8 cho Manhattan (chính) và Gower (đối chứng) với `the current settings in src/library_model.py (PAM, BUILD initialization, seed 42)`.

In [ ]:
metric_paths = run_and_save_metrics(
    df, ROOT / "outputs" / "metrics",
    extra_config={
        "source_filter": ["uci_multi_cleveland"],
        "abnormal_codes": "mọi mã ngoài EXPECTED_VALUES -> NaN -> mode/median",
        "fill_categorical": fill_cat,
        "fill_numeric": fill_num,
    },
)

for name, path in metric_paths.items():
    print(f"{name}: {path}")

pd.read_csv(metric_paths["k_selection"]).round(3)
